In [1]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import os

df = pd.read_csv('/kaggle/input/datasets/atharvadhupkar/filtered-coco-dataset/filtered_coco_metadata.csv')
# print(df.shape)
# print(df.columns.tolist())

In [2]:
import ast

label_cols = ['encoder__name_bus', 'encoder__name_car', 'encoder__name_person', 'encoder__name_truck']
df['label_idx'] = df[label_cols].values.argmax(axis=1)

# Parse the bbox column from string "[x, y, w, h]" into 4 separate columns
df['bbox_parsed'] = df['remainder__bbox'].apply(ast.literal_eval)
df['bbox_x'] = df['bbox_parsed'].apply(lambda b: b[0])
df['bbox_y'] = df['bbox_parsed'].apply(lambda b: b[1])
df['bbox_w'] = df['bbox_parsed'].apply(lambda b: b[2])
df['bbox_h'] = df['bbox_parsed'].apply(lambda b: b[3])

bbox_df = df[['remainder__file_name', 'remainder__coco_url',
              'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'label_idx']].copy()

# Download unique images
os.makedirs('images', exist_ok=True)
url_map = df[['remainder__file_name', 'remainder__coco_url']].drop_duplicates()

for _, row in url_map.iterrows():
    path = f"images/{row['remainder__file_name']}"
    if not os.path.exists(path):
        img = Image.open(BytesIO(requests.get(row['remainder__coco_url']).content)).convert('RGB')
        img.save(path)

print(f"Total crops (bounding boxes): {len(bbox_df)}")
print(f"Unique images: {bbox_df['remainder__file_name'].nunique()}")
print(f"Label distribution:\n{bbox_df['label_idx'].value_counts()}")

Total crops (bounding boxes): 1546
Unique images: 83
Label distribution:
label_idx
2    652
1    434
0    302
3    158
Name: count, dtype: int64


Data augmentation TODO in future done in the following block.

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CocoCropDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"images/{row['remainder__file_name']}").convert('RGB')

        # Crop to the bounding box
        x, y, w, h = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
        # PIL crop takes (left, upper, right, lower)
        img = img.crop((x, y, x + w, y + h))

        if self.transform:
            img = self.transform(img)
        return img, row['label_idx']


dataset = CocoCropDataset(bbox_df, transform)

# 70% train, 15% val, 15% test
train_size = int(0.70 * len(dataset))
val_size   = int(0.15 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=8, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=8, num_workers=2)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Train: 1082, Val: 231, Test: 233


In [4]:
from torchvision import models
import torch
import numpy as np

# Use resnet as a feature extractor (no final layer)
extractor = models.resnet50(weights='IMAGENET1K_V1')
extractor = torch.nn.Sequential(*list(extractor.children())[:-1])
extractor.eval()

def extract_features(loader):
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            feats = extractor(imgs).squeeze(-1).squeeze(-1)
            features.append(feats.numpy())
            labels.append(lbls.numpy())
    return np.concatenate(features), np.concatenate(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test,  y_test  = extract_features(test_loader)

print("features extracted")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


features extracted


In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report

grid = GridSearchCV(
    SVC(class_weight='balanced'),
    {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto']},
    cv=3, scoring='f1_macro', verbose=1
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Val (for hyperparam tuning)")
print(classification_report(y_val, grid.predict(X_val)))

print("Test (final, run once)")
print(classification_report(y_test, grid.predict(X_test)))



Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params: {'C': 1, 'gamma': 'scale'}
Val (for hyperparam tuning)
              precision    recall  f1-score   support

           0       0.68      0.57      0.62        47
           1       0.80      0.66      0.72        61
           2       0.85      0.93      0.89       107
           3       0.30      0.44      0.36        16

    accuracy                           0.75       231
   macro avg       0.66      0.65      0.65       231
weighted avg       0.76      0.75      0.75       231

Test (final, run once)
              precision    recall  f1-score   support

           0       0.54      0.57      0.56        44
           1       0.76      0.67      0.71        63
           2       0.86      0.83      0.84       103
           3       0.42      0.61      0.50        23

    accuracy                           0.71       233
   macro avg       0.65      0.67      0.65       233
weighted avg       0.73      0.71 

In [6]:
# from sklearn.svm import SVC
# from sklearn.metrics import classification_report

# svm = SVC(kernel='rbf', C=1.0, class_weight='balanced')
# svm.fit(X_train, y_train)

# print("Val (for tuning):")
# print(classification_report(y_val, svm.predict(X_val)))

# print("Test (final, run once)")
# print(classification_report(y_test, grid.predict(X_test)))
